In [ ]:
# Core libraries
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.transforms.functional import rotate

# Computer vision & ML
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import albumentations as A
from albumentations.pytorch import ToTensorV2

# YOLO & Object Detection
from ultralytics import YOLO
import ultralytics

# OCR Integration
import easyocr
import pytesseract

# Data handling
import json
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import yaml
from collections import Counter, defaultdict
import random
from datetime import datetime
import pickle
import re
import math

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from matplotlib.patches import Rectangle
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Metrics & Evaluation
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"🎯 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 GPU: {torch.cuda.get_device_name(0)}")
print(f"🎯 Ultralytics version: {ultralytics.__version__}")


In [ ]:
# =============================================================================
# CONFIGURATION SETTINGS
# =============================================================================

class Config:
    """Configuration class for the entire pipeline"""
    
    # Paths
    INPUT_IMAGES_DIR = Path("annotations/input_images")
    JSON_ANNOTATIONS_DIR = Path("annotations/json annotations")
    VISUAL_ANNOTATIONS_DIR = Path("annotations/visual_annotations")
    
    # Output directories
    OUTPUT_DIR = Path("yolo_training_output")
    DATASET_DIR = OUTPUT_DIR / "dataset"
    AUGMENTED_DIR = OUTPUT_DIR / "augmented_data"
    MODELS_DIR = OUTPUT_DIR / "models"
    RESULTS_DIR = OUTPUT_DIR / "results"
    VISUALIZATIONS_DIR = OUTPUT_DIR / "visualizations"
    
    # Character classes mapping (36 classes)
    CHAR_TO_ID = {
        'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7,
        'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 
        'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21,
        'W': 22, 'X': 23, 'Y': 24, 'Z': 25, 
        '0': 26, '1': 27, '2': 28, '3': 29, '4': 30, '5': 31, 
        '6': 32, '7': 33, '8': 34, '9': 35
    }
    
    ID_TO_CHAR = {v: k for k, v in CHAR_TO_ID.items()}
    
    # Serial number patterns (3 formats)
    SERIAL_PATTERNS = {
        'Format1_AAnnnnA': r'^[A-Z]{2}\d{4}[A-Z]$',  # AAnnnnA (e.g., FD8764S)
        'Format2_AAnnnAn': r'^[A-Z]{2}\d{3}[A-Z]\d$', # AAnnnAn (e.g., GF887E1) 
        'Format3_AAAnnnA': r'^[A-Z]{3}\d{3}[A-Z]$'   # AAAnnnA (e.g., JFH772B)
    }
    
    # Training parameters
    TRAIN_SPLIT = 0.8
    VAL_SPLIT = 0.15
    TEST_SPLIT = 0.05
    
    # Augmentation settings
    ROTATION_ANGLES = [-15, -10, -5, 0, 5, 10, 15]  # degrees
    BRIGHTNESS_RANGE = (0.7, 1.3)
    CONTRAST_RANGE = (0.8, 1.2)
    
    # YOLO training parameters
    YOLO_MODEL_SIZE = 'yolov8n.pt'  # nano, small, medium, large, x
    EPOCHS = 100
    BATCH_SIZE = 16
    IMAGE_SIZE = 640
    PATIENCE = 15
    
    # OCR settings
    OCR_CONFIDENCE_THRESHOLD = 0.5
    ENSEMBLE_WEIGHT_YOLO = 0.7
    ENSEMBLE_WEIGHT_OCR = 0.3

# Create output directories
for dir_path in [Config.OUTPUT_DIR, Config.DATASET_DIR, Config.AUGMENTED_DIR, 
                 Config.MODELS_DIR, Config.RESULTS_DIR, Config.VISUALIZATIONS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("✅ Configuration initialized")
print(f"📊 Total character classes: {len(Config.CHAR_TO_ID)}")
print(f"📊 Serial number formats: {len(Config.SERIAL_PATTERNS)}")
print(f"📁 Output directory: {Config.OUTPUT_DIR.absolute()}")


In [ ]:
class DataAnalyzer:
    """Comprehensive data analysis for the annotation dataset"""
    
    def __init__(self):
        self.annotations_data = []
        self.character_distribution = Counter()
        self.pattern_distribution = Counter()
        self.image_stats = {}
        
    def load_all_annotations(self):
        """Load and analyze all JSON annotation files"""
        print("🔍 Loading annotation files...")
        
        json_files = list(Config.JSON_ANNOTATIONS_DIR.glob("*.json"))
        print(f"📁 Found {len(json_files)} annotation files")
        
        for json_file in json_files:
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                    
                # Extract image info
                filename = data['image_info']['filename']
                width = data['image_info']['width']
                height = data['image_info']['height']
                
                # Extract character annotations
                char_annotations = []
                if 'character_annotations' in data:
                    char_annotations = data['character_annotations']
                elif 'validated_words' in data and len(data['validated_words']) > 0:
                    # Fallback to validated_words character breakdown
                    for word_data in data['validated_words']:
                        if 'character_annotations' in word_data:
                            char_annotations.extend(word_data['character_annotations'])
                
                # Count characters
                for char_data in char_annotations:
                    char = char_data.get('text', '').upper()
                    if char in Config.CHAR_TO_ID:
                        self.character_distribution[char] += 1
                
                # Determine pattern
                serial_number = filename.split('.')[0]  # Remove extension
                pattern = self._classify_pattern(serial_number)
                self.pattern_distribution[pattern] += 1
                
                # Store data
                annotation_entry = {
                    'filename': filename,
                    'json_file': json_file.name,
                    'width': width,
                    'height': height,
                    'character_count': len(char_annotations),
                    'characters': [c.get('text', '').upper() for c in char_annotations],
                    'pattern': pattern,
                    'serial_number': serial_number
                }
                
                self.annotations_data.append(annotation_entry)
                
            except Exception as e:
                print(f"⚠️ Error loading {json_file.name}: {e}")
        
        print(f"✅ Loaded {len(self.annotations_data)} annotations successfully")
        return self.annotations_data
    
    def _classify_pattern(self, serial_number):
        """Classify serial number into one of the 3 patterns"""
        for pattern_name, regex in Config.SERIAL_PATTERNS.items():
            if re.match(regex, serial_number):
                return pattern_name
        return "Unknown"
    
    def generate_comprehensive_report(self):
        """Generate comprehensive data analysis report"""
        if not self.annotations_data:
            self.load_all_annotations()
        
        df = pd.DataFrame(self.annotations_data)
        
        print("=" * 70)
        print("📊 COMPREHENSIVE DATA ANALYSIS REPORT")
        print("=" * 70)
        
        # Basic statistics
        total_images = len(df)
        total_characters = sum(df['character_count'])
        avg_chars_per_image = total_characters / total_images if total_images > 0 else 0
        
        print(f"📁 Total Images: {total_images}")
        print(f"🔤 Total Characters: {total_characters}")
        print(f"📊 Average Characters per Image: {avg_chars_per_image:.2f}")
        print(f"📏 Image Dimensions Range: {df['width'].min()}x{df['height'].min()} to {df['width'].max()}x{df['height'].max()}")
        
        # Pattern distribution
        print(f"\n📋 Serial Number Pattern Distribution:")
        for pattern, count in self.pattern_distribution.most_common():
            percentage = (count / total_images) * 100
            print(f"   {pattern}: {count} ({percentage:.1f}%)\")\n        
        # Character distribution
        print(f"🔤 Character Class Distribution (Top 20):")
        for char, count in self.character_distribution.most_common(20):
            class_id = Config.CHAR_TO_ID.get(char, -1)
            percentage = (count / total_characters) * 100
            print(f"   {char} (ID:{class_id:2d}): {count:3d} ({percentage:4.1f}%)")
        
        return df
    
    def visualize_distributions(self):
        """Create comprehensive visualizations"""
        if not self.annotations_data:
            self.load_all_annotations()
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('📊 Dataset Analysis Dashboard', fontsize=16, fontweight='bold')
        
        # 1. Character distribution
        chars = list(self.character_distribution.keys())
        counts = list(self.character_distribution.values())
        
        ax1 = axes[0, 0]
        bars = ax1.bar(chars, counts, color='skyblue', alpha=0.8)
        ax1.set_title('Character Distribution (All 36 Classes)', fontweight='bold')
        ax1.set_xlabel('Character Classes')
        ax1.set_ylabel('Frequency')
        ax1.tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.5, f'{int(height)}',
                    ha='center', va='bottom', fontsize=8)
        
        # 2. Pattern distribution
        patterns = list(self.pattern_distribution.keys())
        pattern_counts = list(self.pattern_distribution.values())
        
        ax2 = axes[0, 1]
        colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
        wedges, texts, autotexts = ax2.pie(pattern_counts, labels=patterns, autopct='%1.1f%%',
                                          colors=colors, startangle=90)
        ax2.set_title('Serial Number Pattern Distribution', fontweight='bold')
        
        # 3. Characters per image distribution
        df = pd.DataFrame(self.annotations_data)
        char_counts = df['character_count'].value_counts().sort_index()
        
        ax3 = axes[1, 0]
        ax3.bar(char_counts.index, char_counts.values, color='lightcoral', alpha=0.8)
        ax3.set_title('Characters per Image Distribution', fontweight='bold')
        ax3.set_xlabel('Number of Characters')
        ax3.set_ylabel('Number of Images')
        
        # 4. Image dimension distribution
        ax4 = axes[1, 1]
        ax4.scatter(df['width'], df['height'], alpha=0.6, c='purple')
        ax4.set_title('Image Dimensions Distribution', fontweight='bold')
        ax4.set_xlabel('Width (pixels)')
        ax4.set_ylabel('Height (pixels)')
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(Config.VISUALIZATIONS_DIR / 'dataset_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        return fig

# Initialize and run analysis
analyzer = DataAnalyzer()
df_annotations = analyzer.generate_comprehensive_report()
analyzer.visualize_distributions()


In [ ]:
class YOLODatasetConverter:
    """Convert annotation dataset to YOLO format with train/val/test splits"""
    
    def __init__(self):
        self.train_images = []
        self.val_images = []
        self.test_images = []
        
    def create_yolo_dataset_structure(self):
        """Create YOLO dataset directory structure"""
        print("🏗️ Creating YOLO dataset structure...")
        
        # Create directories
        for split in ['train', 'val', 'test']:
            (Config.DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
            (Config.DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
        
        print(f"✅ Dataset structure created at {Config.DATASET_DIR}")
    
    def split_dataset(self, annotations_df):
        """Smart dataset splitting based on patterns"""
        print("📊 Splitting dataset strategically...")
        
        # Group by pattern for stratified split
        pattern_groups = annotations_df.groupby('pattern')
        
        train_list, val_list, test_list = [], [], []
        
        for pattern, group in pattern_groups:
            # Shuffle within each pattern
            group_shuffled = group.sample(frac=1, random_state=42)
            n = len(group_shuffled)
            
            # Calculate splits
            train_size = int(n * Config.TRAIN_SPLIT)
            val_size = int(n * Config.VAL_SPLIT)
            
            # Split the group
            train_group = group_shuffled.iloc[:train_size]
            val_group = group_shuffled.iloc[train_size:train_size + val_size]
            test_group = group_shuffled.iloc[train_size + val_size:]
            
            train_list.append(train_group)
            val_list.append(val_group)
            if len(test_group) > 0:
                test_list.append(test_group)
            
            print(f"   {pattern}: Train={len(train_group)}, Val={len(val_group)}, Test={len(test_group)}")
        
        # Combine all splits
        train_df = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()
        val_df = pd.concat(val_list, ignore_index=True) if val_list else pd.DataFrame()
        test_df = pd.concat(test_list, ignore_index=True) if test_list else pd.DataFrame()
        
        print(f"📊 Final split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
        
        return train_df, val_df, test_df
    
    def convert_annotations_to_yolo(self, split_name, df_split):
        """Convert JSON annotations to YOLO format"""
        print(f"🔄 Converting {split_name} annotations to YOLO format...")
        
        converted_count = 0
        
        for _, row in df_split.iterrows():
            try:
                # Load original JSON annotation
                json_path = Config.JSON_ANNOTATIONS_DIR / row['json_file']
                with open(json_path, 'r') as f:
                    data = json.load(f)
                
                # Get image info
                image_width = data['image_info']['width']
                image_height = data['image_info']['height']
                filename = row['filename']
                
                # Copy image to dataset directory
                src_image = Config.INPUT_IMAGES_DIR / filename
                dst_image = Config.DATASET_DIR / split_name / 'images' / filename
                
                if src_image.exists():
                    shutil.copy2(src_image, dst_image)
                else:
                    print(f"⚠️ Warning: Image not found: {src_image}")
                    continue
                
                # Extract character annotations
                char_annotations = []
                if 'character_annotations' in data:
                    char_annotations = data['character_annotations']
                elif 'validated_words' in data and len(data['validated_words']) > 0:
                    # Extract from validated_words if available
                    for word_data in data['validated_words']:
                        if 'character_annotations' in word_data:
                            char_annotations.extend(word_data['character_annotations'])
                
                # Create YOLO label file
                label_path = Config.DATASET_DIR / split_name / 'labels' / f"{Path(filename).stem}.txt"
                
                with open(label_path, 'w') as f:
                    for char_data in char_annotations:
                        char = char_data.get('text', '').upper()
                        if char in Config.CHAR_TO_ID:
                            class_id = Config.CHAR_TO_ID[char]
                            
                            # Get bounding box
                            bbox = char_data.get('bbox', [])
                            if len(bbox) == 4:
                                x_min, y_min, x_max, y_max = bbox
                                
                                # Convert to YOLO format (normalized center coordinates)
                                center_x = (x_min + x_max) / 2 / image_width
                                center_y = (y_min + y_max) / 2 / image_height
                                width = (x_max - x_min) / image_width
                                height = (y_max - y_min) / image_height
                                
                                # Ensure values are within [0, 1]
                                center_x = max(0, min(1, center_x))
                                center_y = max(0, min(1, center_y))
                                width = max(0, min(1, width))
                                height = max(0, min(1, height))
                                
                                f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}\n")
                
                converted_count += 1
                
            except Exception as e:
                print(f"⚠️ Error converting {row['filename']}: {e}")
        
        print(f"✅ Converted {converted_count}/{len(df_split)} {split_name} annotations")
        return converted_count
    
    def create_dataset_yaml(self):
        """Create dataset.yaml for YOLO training"""
        dataset_config = {
            'path': str(Config.DATASET_DIR.absolute()),
            'train': 'train/images',
            'val': 'val/images',
            'test': 'test/images',
            'nc': len(Config.CHAR_TO_ID),  # number of classes
            'names': Config.ID_TO_CHAR
        }
        
        yaml_path = Config.DATASET_DIR / 'dataset.yaml'
        with open(yaml_path, 'w') as f:
            yaml.dump(dataset_config, f, default_flow_style=False)
        
        print(f"✅ Created dataset.yaml at {yaml_path}")
        return yaml_path
    
    def convert_full_dataset(self, annotations_df):
        """Complete dataset conversion pipeline"""
        print("🚀 Starting full dataset conversion...")
        
        # Create directory structure
        self.create_yolo_dataset_structure()
        
        # Split dataset
        train_df, val_df, test_df = self.split_dataset(annotations_df)
        
        # Convert each split
        train_count = self.convert_annotations_to_yolo('train', train_df)
        val_count = self.convert_annotations_to_yolo('val', val_df)
        test_count = self.convert_annotations_to_yolo('test', test_df) if len(test_df) > 0 else 0
        
        # Create dataset.yaml
        yaml_path = self.create_dataset_yaml()
        
        # Summary
        print("=" * 60)
        print("📊 DATASET CONVERSION SUMMARY")
        print("=" * 60)
        print(f"✅ Train: {train_count} images")
        print(f"✅ Validation: {val_count} images") 
        print(f"✅ Test: {test_count} images")
        print(f"📁 Dataset location: {Config.DATASET_DIR}")
        print(f"📄 Configuration: {yaml_path}")
        print("=" * 60)
        
        return {
            'train_count': train_count,
            'val_count': val_count,
            'test_count': test_count,
            'yaml_path': yaml_path
        }

# Convert dataset
converter = YOLODatasetConverter()
conversion_results = converter.convert_full_dataset(df_annotations)


In [ ]:
class AdvancedDataAugmentor:
    """Advanced data augmentation with precise coordinate transformation"""
    
    def __init__(self):
        self.augmentation_stats = {
            'total_original': 0,
            'total_augmented': 0,
            'by_angle': defaultdict(int),
            'by_brightness': defaultdict(int)
        }
    
    def rotate_image_and_bboxes(self, image, bboxes, angle):
        """Rotate image and adjust bounding box coordinates accordingly"""
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        
        # Create rotation matrix
        rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        
        # Calculate new image size
        cos_val = np.abs(rotation_matrix[0, 0])
        sin_val = np.abs(rotation_matrix[0, 1])
        new_w = int((h * sin_val) + (w * cos_val))
        new_h = int((h * cos_val) + (w * sin_val))
        
        # Adjust rotation matrix for new center
        rotation_matrix[0, 2] += (new_w / 2) - center[0]
        rotation_matrix[1, 2] += (new_h / 2) - center[1]
        
        # Rotate image
        rotated_image = cv2.warpAffine(image, rotation_matrix, (new_w, new_h), 
                                     flags=cv2.INTER_LINEAR, 
                                     borderMode=cv2.BORDER_CONSTANT,
                                     borderValue=(0, 0, 0))
        
        # Transform bounding boxes
        rotated_bboxes = []
        for bbox in bboxes:
            # Convert YOLO format to corner coordinates
            center_x, center_y, width, height, class_id = bbox
            
            # Convert to absolute coordinates
            abs_center_x = center_x * w
            abs_center_y = center_y * h
            abs_width = width * w
            abs_height = height * h
            
            # Get corner points
            x1 = abs_center_x - abs_width / 2
            y1 = abs_center_y - abs_height / 2
            x2 = abs_center_x + abs_width / 2
            y2 = abs_center_y + abs_height / 2
            
            # Transform corners
            corners = np.array([[x1, y1, 1], [x2, y1, 1], [x2, y2, 1], [x1, y2, 1]]).T
            rotated_corners = rotation_matrix @ corners
            
            # Get new bounding box
            x_coords = rotated_corners[0, :]
            y_coords = rotated_corners[1, :]
            
            new_x1, new_x2 = np.min(x_coords), np.max(x_coords)
            new_y1, new_y2 = np.min(y_coords), np.max(y_coords)
            
            # Convert back to YOLO format (normalized)
            new_center_x = (new_x1 + new_x2) / 2 / new_w
            new_center_y = (new_y1 + new_y2) / 2 / new_h
            new_width = (new_x2 - new_x1) / new_w
            new_height = (new_y2 - new_y1) / new_h
            
            # Ensure values are within [0, 1]
            new_center_x = max(0, min(1, new_center_x))
            new_center_y = max(0, min(1, new_center_y))
            new_width = max(0, min(1, new_width))
            new_height = max(0, min(1, new_height))
            
            rotated_bboxes.append([new_center_x, new_center_y, new_width, new_height, class_id])
        
        return rotated_image, rotated_bboxes, (new_w, new_h)
    
    def adjust_brightness_contrast(self, image, brightness=1.0, contrast=1.0):
        """Adjust image brightness and contrast"""
        # Convert to float for calculations
        image_float = image.astype(np.float32)
        
        # Apply brightness and contrast
        adjusted = image_float * contrast + (brightness - 1.0) * 255
        
        # Clip values and convert back to uint8
        adjusted = np.clip(adjusted, 0, 255).astype(np.uint8)
        
        return adjusted
    
    def load_yolo_annotations(self, label_path):
        """Load YOLO format annotations"""
        bboxes = []
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    values = line.strip().split()
                    if len(values) == 5:
                        class_id = int(values[0])
                        center_x = float(values[1])
                        center_y = float(values[2])
                        width = float(values[3])
                        height = float(values[4])
                        bboxes.append([center_x, center_y, width, height, class_id])
        return bboxes
    
    def save_yolo_annotations(self, bboxes, output_path):
        """Save YOLO format annotations"""
        with open(output_path, 'w') as f:
            for bbox in bboxes:
                class_id = int(bbox[4])
                center_x, center_y, width, height = bbox[:4]
                f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}\\n")
    
    def augment_dataset_split(self, split_name, augment_factor=3):
        """Augment a specific dataset split"""
        print(f"🔄 Augmenting {split_name} dataset...")
        
        images_dir = Config.DATASET_DIR / split_name / 'images'
        labels_dir = Config.DATASET_DIR / split_name / 'labels'
        
        # Create augmented directories
        aug_images_dir = Config.AUGMENTED_DIR / split_name / 'images'
        aug_labels_dir = Config.AUGMENTED_DIR / split_name / 'labels'
        aug_images_dir.mkdir(parents=True, exist_ok=True)
        aug_labels_dir.mkdir(parents=True, exist_ok=True)
        
        # Get all image files
        image_files = list(images_dir.glob('*.jpg'))
        total_originals = len(image_files)
        augmented_count = 0
        
        for img_path in image_files:
            try:
                # Load image
                image = cv2.imread(str(img_path))
                if image is None:
                    continue
                
                # Load corresponding annotations
                label_path = labels_dir / f"{img_path.stem}.txt"
                bboxes = self.load_yolo_annotations(label_path)
                
                if not bboxes:  # Skip images without annotations
                    continue
                
                # Copy original to augmented directory
                orig_img_dest = aug_images_dir / img_path.name
                orig_label_dest = aug_labels_dir / f"{img_path.stem}.txt"
                shutil.copy2(img_path, orig_img_dest)
                self.save_yolo_annotations(bboxes, orig_label_dest)
                
                # Generate augmentations
                for i, angle in enumerate(Config.ROTATION_ANGLES):
                    if angle == 0:  # Skip original (already copied)
                        continue
                    
                    # Generate multiple brightness/contrast combinations
                    brightness_vals = np.linspace(*Config.BRIGHTNESS_RANGE, 3)
                    contrast_vals = np.linspace(*Config.CONTRAST_RANGE, 3)
                    
                    for j, (brightness, contrast) in enumerate(zip(brightness_vals, contrast_vals)):
                        if augmented_count >= total_originals * augment_factor:
                            break
                        
                        # Apply rotation
                        rotated_img, rotated_bboxes, new_size = self.rotate_image_and_bboxes(
                            image, bboxes, angle
                        )
                        
                        # Apply brightness/contrast
                        final_img = self.adjust_brightness_contrast(
                            rotated_img, brightness, contrast
                        )
                        
                        # Save augmented image and annotations
                        aug_filename = f"{img_path.stem}_rot{angle:+03d}_br{brightness:.2f}_co{contrast:.2f}.jpg"
                        aug_img_path = aug_images_dir / aug_filename
                        aug_label_path = aug_labels_dir / f"{Path(aug_filename).stem}.txt"
                        
                        cv2.imwrite(str(aug_img_path), final_img)
                        self.save_yolo_annotations(rotated_bboxes, aug_label_path)
                        
                        augmented_count += 1
                        
                        # Update stats
                        self.augmentation_stats['by_angle'][angle] += 1
                        self.augmentation_stats['by_brightness'][f"{brightness:.1f}"] += 1
                
            except Exception as e:
                print(f"⚠️ Error augmenting {img_path.name}: {e}")
        
        self.augmentation_stats['total_original'] += total_originals
        self.augmentation_stats['total_augmented'] += augmented_count
        
        print(f"✅ {split_name}: {total_originals} original → {augmented_count} augmented")
        return augmented_count
    
    def create_augmented_dataset_yaml(self):
        """Create dataset.yaml for augmented dataset"""
        dataset_config = {
            'path': str(Config.AUGMENTED_DIR.absolute()),
            'train': 'train/images',
            'val': 'val/images',
            'test': 'test/images',
            'nc': len(Config.CHAR_TO_ID),
            'names': Config.ID_TO_CHAR
        }
        
        yaml_path = Config.AUGMENTED_DIR / 'dataset.yaml'
        with open(yaml_path, 'w') as f:
            yaml.dump(dataset_config, f, default_flow_style=False)
        
        return yaml_path
    
    def visualize_augmentation_sample(self, sample_image_name):
        """Visualize augmentation effects on a sample image"""
        # Find the image in train directory
        orig_img_path = Config.DATASET_DIR / 'train' / 'images' / sample_image_name
        orig_label_path = Config.DATASET_DIR / 'train' / 'labels' / f"{Path(sample_image_name).stem}.txt"
        
        if not orig_img_path.exists():
            print(f"⚠️ Sample image not found: {sample_image_name}")
            return
        
        # Load original
        orig_img = cv2.imread(str(orig_img_path))
        orig_bboxes = self.load_yolo_annotations(orig_label_path)
        
        # Create visualization
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        fig.suptitle(f'Data Augmentation Visualization: {sample_image_name}', fontsize=16)
        
        # Original image
        axes[0, 0].imshow(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB))
        axes[0, 0].set_title('Original')
        axes[0, 0].axis('off')
        
        # Show different rotations
        angles_to_show = [-15, -5, 5, 15]
        for i, angle in enumerate(angles_to_show):
            row, col = divmod(i + 1, 4)
            
            rotated_img, rotated_bboxes, _ = self.rotate_image_and_bboxes(
                orig_img, orig_bboxes, angle
            )
            
            axes[row, col].imshow(cv2.cvtColor(rotated_img, cv2.COLOR_BGR2RGB))
            axes[row, col].set_title(f'Rotated {angle}°')
            axes[row, col].axis('off')
        
        # Show brightness variations
        brightness_vals = [0.7, 0.85, 1.15, 1.3]
        for i, brightness in enumerate(brightness_vals):
            row, col = 1, i
            
            bright_img = self.adjust_brightness_contrast(orig_img, brightness=brightness)
            axes[row, col].imshow(cv2.cvtColor(bright_img, cv2.COLOR_BGR2RGB))
            axes[row, col].set_title(f'Brightness {brightness}')
            axes[row, col].axis('off')
        
        plt.tight_layout()
        plt.savefig(Config.VISUALIZATIONS_DIR / 'augmentation_samples.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def run_full_augmentation(self):
        """Run complete data augmentation pipeline"""
        print("🚀 Starting comprehensive data augmentation...")
        
        # Augment each split
        train_aug = self.augment_dataset_split('train', augment_factor=4)
        val_aug = self.augment_dataset_split('val', augment_factor=2)
        
        # Create augmented dataset yaml
        aug_yaml = self.create_augmented_dataset_yaml()
        
        # Print summary
        print("\\n" + "=" * 60)
        print("📊 DATA AUGMENTATION SUMMARY")
        print("=" * 60)
        print(f"🔢 Original images: {self.augmentation_stats['total_original']}")
        print(f"🔄 Augmented images: {self.augmentation_stats['total_augmented']}")
        print(f"📈 Augmentation ratio: {self.augmentation_stats['total_augmented'] / self.augmentation_stats['total_original']:.1f}x")
        print(f"📁 Augmented dataset: {Config.AUGMENTED_DIR}")
        print(f"📄 Configuration: {aug_yaml}")
        print("=" * 60)
        
        return {
            'total_original': self.augmentation_stats['total_original'],
            'total_augmented': self.augmentation_stats['total_augmented'],
            'yaml_path': aug_yaml
        }

# Run data augmentation
augmentor = AdvancedDataAugmentor()
aug_results = augmentor.run_full_augmentation()

# Visualize augmentation on a sample image
sample_files = list((Config.DATASET_DIR / 'train' / 'images').glob('*.jpg'))
if sample_files:
    augmentor.visualize_augmentation_sample(sample_files[0].name)


In [ ]:
class YOLOv8Trainer:
    """Advanced YOLOv8 training with comprehensive monitoring"""
    
    def __init__(self, use_augmented_data=True):
        self.use_augmented = use_augmented_data
        self.dataset_path = Config.AUGMENTED_DIR if use_augmented_data else Config.DATASET_DIR
        self.model = None
        self.training_results = None
        
    def setup_training_environment(self):
        """Setup optimal training environment"""
        print("⚙️ Setting up training environment...")
        
        # Check CUDA
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"🎯 Training device: {device}")
        
        if torch.cuda.is_available():
            print(f"🎯 GPU: {torch.cuda.get_device_name(0)}")
            print(f"🎯 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        
        # Initialize model
        print(f"📥 Loading {Config.YOLO_MODEL_SIZE} model...")
        self.model = YOLO(Config.YOLO_MODEL_SIZE)
        
        return device
    
    def train_yolo_model(self):
        """Train YOLOv8 model with optimal parameters"""
        device = self.setup_training_environment()
        
        print(f"🚀 Starting YOLOv8 training...")
        print(f"📊 Dataset: {'Augmented' if self.use_augmented else 'Original'}")
        print(f"📊 Epochs: {Config.EPOCHS}")
        print(f"📊 Batch size: {Config.BATCH_SIZE}")
        print(f"📊 Image size: {Config.IMAGE_SIZE}")
        
        # Training parameters optimized for character detection
        training_args = {
            'data': str(self.dataset_path / 'dataset.yaml'),
            'epochs': Config.EPOCHS,
            'imgsz': Config.IMAGE_SIZE,
            'batch': Config.BATCH_SIZE,
            'name': f'character_detection_{"aug" if self.use_augmented else "orig"}',
            'project': str(Config.MODELS_DIR),
            'patience': Config.PATIENCE,
            'save_period': 10,
            'device': device,
            'workers': 4,
            'exist_ok': True,
            'pretrained': True,
            'optimizer': 'AdamW',
            'lr0': 0.01,
            'momentum': 0.937,
            'weight_decay': 0.0005,
            'warmup_epochs': 5,
            'warmup_momentum': 0.8,
            'warmup_bias_lr': 0.1,
            'cls': 0.5,
            'box': 7.5,
            'dfl': 1.5,
            'pose': 12.0,
            'kobj': 1.0,
            'plots': True,
            'save': True,
            'verbose': True,
            'deterministic': False,
            'single_cls': False,
            'rect': False,
            'cos_lr': False,
            'close_mosaic': 10,
            'resume': False,
            'amp': True,  # Automatic Mixed Precision
            'fraction': 1.0,
            'profile': False,
            'freeze': None,
            'multi_scale': False,
            'overlap_mask': True,
            'mask_ratio': 4,
            'dropout': 0.0,
            'val': True,
            'split': 'val',
            'save_json': True,
            'save_hybrid': False,
            'conf': None,
            'iou': 0.7,
            'max_det': 300,
            'half': False,
            'dnn': False,
            'plots': True
        }
        
        # Start training
        self.training_results = self.model.train(**training_args)
        
        print("✅ Training completed!")
        return self.training_results
    
    def analyze_training_results(self):
        """Comprehensive analysis of training results"""
        if not self.training_results:
            print("⚠️ No training results to analyze")
            return
        
        results_dir = self.training_results.save_dir
        print(f"📊 Analyzing results from: {results_dir}")
        
        # Load training metrics
        try:
            results_csv = results_dir / 'results.csv'
            if results_csv.exists():
                df_results = pd.read_csv(results_csv)
                
                # Create comprehensive training visualization
                self.create_training_dashboard(df_results, results_dir)
                
                # Print key metrics
                final_metrics = df_results.iloc[-1]
                print("\\n" + "=" * 60)
                print("📊 FINAL TRAINING METRICS")
                print("=" * 60)
                print(f"📈 mAP50: {final_metrics.get('metrics/mAP50(B)', 'N/A'):.4f}")
                print(f"📈 mAP50-95: {final_metrics.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
                print(f"📉 Box Loss: {final_metrics.get('train/box_loss', 'N/A'):.4f}")
                print(f"📉 Class Loss: {final_metrics.get('train/cls_loss', 'N/A'):.4f}")
                print(f"📉 DFL Loss: {final_metrics.get('train/dfl_loss', 'N/A'):.4f}")
                print(f"🎯 Precision: {final_metrics.get('metrics/precision(B)', 'N/A'):.4f}")
                print(f"🎯 Recall: {final_metrics.get('metrics/recall(B)', 'N/A'):.4f}")
                print("=" * 60)
            
        except Exception as e:
            print(f"⚠️ Error analyzing results: {e}")
    
    def create_training_dashboard(self, df_results, results_dir):
        """Create comprehensive training dashboard"""
        
        fig, axes = plt.subplots(3, 2, figsize=(16, 18))
        fig.suptitle('🎯 YOLOv8 Training Dashboard', fontsize=16, fontweight='bold')
        
        epochs = range(1, len(df_results) + 1)
        
        # 1. Loss curves
        axes[0, 0].plot(epochs, df_results.get('train/box_loss', []), 'b-', label='Box Loss', linewidth=2)
        axes[0, 0].plot(epochs, df_results.get('train/cls_loss', []), 'r-', label='Class Loss', linewidth=2)
        axes[0, 0].plot(epochs, df_results.get('train/dfl_loss', []), 'g-', label='DFL Loss', linewidth=2)
        axes[0, 0].set_title('Training Losses', fontweight='bold')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Validation losses
        axes[0, 1].plot(epochs, df_results.get('val/box_loss', []), 'b--', label='Val Box Loss', linewidth=2)
        axes[0, 1].plot(epochs, df_results.get('val/cls_loss', []), 'r--', label='Val Class Loss', linewidth=2)
        axes[0, 1].plot(epochs, df_results.get('val/dfl_loss', []), 'g--', label='Val DFL Loss', linewidth=2)
        axes[0, 1].set_title('Validation Losses', fontweight='bold')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. mAP metrics
        axes[1, 0].plot(epochs, df_results.get('metrics/mAP50(B)', []), 'purple', linewidth=3, label='mAP@0.5')
        axes[1, 0].plot(epochs, df_results.get('metrics/mAP50-95(B)', []), 'orange', linewidth=3, label='mAP@0.5:0.95')
        axes[1, 0].set_title('Mean Average Precision (mAP)', fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('mAP')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Precision and Recall
        axes[1, 1].plot(epochs, df_results.get('metrics/precision(B)', []), 'darkgreen', linewidth=2, label='Precision')
        axes[1, 1].plot(epochs, df_results.get('metrics/recall(B)', []), 'darkred', linewidth=2, label='Recall')
        axes[1, 1].set_title('Precision & Recall', fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Score')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        # 5. Learning rate
        if 'lr/pg0' in df_results.columns:
            axes[2, 0].plot(epochs, df_results['lr/pg0'], 'brown', linewidth=2)
            axes[2, 0].set_title('Learning Rate Schedule', fontweight='bold')
            axes[2, 0].set_xlabel('Epoch')
            axes[2, 0].set_ylabel('Learning Rate')
            axes[2, 0].grid(True, alpha=0.3)
        
        # 6. Model size and speed metrics
        if 'x/lr0' in df_results.columns:
            axes[2, 1].plot(epochs, df_results.get('x/lr0', []), 'teal', linewidth=2)
            axes[2, 1].set_title('Optimization Metrics', fontweight='bold')
            axes[2, 1].set_xlabel('Epoch')
            axes[2, 1].set_ylabel('Value')
            axes[2, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(Config.VISUALIZATIONS_DIR / 'training_dashboard.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        return fig
    
    def export_model_formats(self):
        """Export model to different formats for deployment"""
        if not self.model:
            print("⚠️ No model to export")
            return
        
        print("📦 Exporting model to different formats...")
        
        # Get best model path
        model_dir = self.training_results.save_dir
        best_model_path = model_dir / 'weights' / 'best.pt'
        
        if not best_model_path.exists():
            print(f"⚠️ Best model not found at {best_model_path}")
            return
        
        # Load best model
        best_model = YOLO(str(best_model_path))
        
        export_results = {}
        
        # Export to ONNX (recommended for deployment)
        try:
            onnx_path = best_model.export(format='onnx', imgsz=Config.IMAGE_SIZE)
            export_results['onnx'] = onnx_path
            print(f"✅ ONNX model exported: {onnx_path}")
        except Exception as e:
            print(f"⚠️ ONNX export failed: {e}")
        
        # Export to TensorRT (if available)
        if torch.cuda.is_available():
            try:
                engine_path = best_model.export(format='engine', imgsz=Config.IMAGE_SIZE)
                export_results['tensorrt'] = engine_path
                print(f"✅ TensorRT model exported: {engine_path}")
            except Exception as e:
                print(f"⚠️ TensorRT export failed: {e}")
        
        # Export to TorchScript
        try:
            torchscript_path = best_model.export(format='torchscript', imgsz=Config.IMAGE_SIZE)
            export_results['torchscript'] = torchscript_path
            print(f"✅ TorchScript model exported: {torchscript_path}")
        except Exception as e:
            print(f"⚠️ TorchScript export failed: {e}")
        
        return export_results

# Initialize and run training
print("🎯 Initializing YOLOv8 Trainer...")
trainer = YOLOv8Trainer(use_augmented_data=True)

# Train the model
training_results = trainer.train_yolo_model()

# Analyze results
trainer.analyze_training_results()

# Export model for deployment
export_results = trainer.export_model_formats()


In [ ]:
class HybridOCRYOLOPredictor:
    """Advanced hybrid approach combining YOLO detection with OCR validation"""
    
    def __init__(self, yolo_model_path=None):
        # Initialize OCR readers
        self.easyocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
        
        # Load trained YOLO model
        if yolo_model_path and Path(yolo_model_path).exists():
            self.yolo_model = YOLO(yolo_model_path)
            print(f"✅ YOLO model loaded: {yolo_model_path}")
        else:
            print("⚠️ YOLO model not provided, using detection results only")
            self.yolo_model = None
        
        # Serial patterns for validation
        self.serial_patterns = [
            r'^[A-Z]{2}\\d{4}[A-Z]$',  # AAnnnnA
            r'^[A-Z]{2}\\d{3}[A-Z]\\d$', # AAnnnAn  
            r'^[A-Z]{3}\\d{3}[A-Z]$'   # AAAnnnA
        ]
    
    def preprocess_image_for_ocr(self, image):
        """Enhanced preprocessing for better OCR accuracy"""
        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        
        # Apply adaptive thresholding
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                     cv2.THRESH_BINARY, 11, 2)
        
        # Morphological operations to clean up
        kernel = np.ones((2,2), np.uint8)
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        
        # Resize for better OCR if needed
        h, w = cleaned.shape
        if max(h, w) < 100:
            scale = 200 / max(h, w)
            new_w, new_h = int(w * scale), int(h * scale)
            cleaned = cv2.resize(cleaned, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
        
        return cleaned
    
    def run_easyocr_detection(self, image):
        """Run EasyOCR detection and character recognition"""
        results = self.easyocr_reader.readtext(image, detail=1, paragraph=False)
        
        detections = []
        for bbox, text, confidence in results:
            if confidence > Config.OCR_CONFIDENCE_THRESHOLD:
                # Convert bbox format
                bbox_np = np.array(bbox, dtype=np.int32)
                x_min, y_min = bbox_np[:, 0].min(), bbox_np[:, 1].min()
                x_max, y_max = bbox_np[:, 0].max(), bbox_np[:, 1].max()
                
                detections.append({
                    'bbox': [x_min, y_min, x_max, y_max],
                    'text': text.strip().upper(),
                    'confidence': confidence,
                    'method': 'EasyOCR'
                })
        
        return detections
    
    def run_yolo_detection(self, image):
        """Run YOLO character detection"""
        if not self.yolo_model:
            return []
        
        try:
            # Run inference
            results = self.yolo_model(image, conf=0.3, iou=0.7)
            
            detections = []
            for result in results:
                boxes = result.boxes
                if boxes is not None:
                    for i, box in enumerate(boxes):
                        # Extract box coordinates
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        confidence = float(box.conf[0].cpu().numpy())
                        class_id = int(box.cls[0].cpu().numpy())
                        
                        # Convert class ID to character
                        character = Config.ID_TO_CHAR.get(class_id, '?')
                        
                        detections.append({
                            'bbox': [int(x1), int(y1), int(x2), int(y2)],
                            'text': character,
                            'confidence': confidence,
                            'class_id': class_id,
                            'method': 'YOLO'
                        })
            
            # Sort detections by x-coordinate for proper order
            detections.sort(key=lambda x: x['bbox'][0])
            
            return detections
            
        except Exception as e:
            print(f"⚠️ YOLO detection error: {e}")
            return []
    
    def merge_detections(self, ocr_detections, yolo_detections, image_shape):
        """Intelligently merge OCR and YOLO detections"""
        merged_results = []
        
        # If YOLO found character-level detections, prioritize those
        if yolo_detections:
            # Group YOLO detections by proximity (likely same word)
            word_groups = self.group_nearby_detections(yolo_detections, threshold=50)
            
            for group in word_groups:
                # Sort characters in group by x-coordinate
                group.sort(key=lambda x: x['bbox'][0])
                
                # Reconstruct word from characters
                word_text = ''.join([det['text'] for det in group])
                
                # Calculate word bounding box
                x_coords = [det['bbox'][0] for det in group] + [det['bbox'][2] for det in group]
                y_coords = [det['bbox'][1] for det in group] + [det['bbox'][3] for det in group]
                
                word_bbox = [min(x_coords), min(y_coords), max(x_coords), max(y_coords)]
                
                # Calculate average confidence
                avg_confidence = np.mean([det['confidence'] for det in group])
                
                merged_results.append({
                    'bbox': word_bbox,
                    'text': word_text,
                    'confidence': avg_confidence * Config.ENSEMBLE_WEIGHT_YOLO,
                    'method': 'YOLO_Merged',
                    'character_detections': group
                })
        
        # Add OCR detections with adjusted confidence
        for ocr_det in ocr_detections:
            # Check if OCR detection overlaps significantly with YOLO
            overlaps = False
            for merged_det in merged_results:
                if self.calculate_iou(ocr_det['bbox'], merged_det['bbox']) > 0.3:
                    overlaps = True
                    break
            
            if not overlaps:
                ocr_det_copy = ocr_det.copy()
                ocr_det_copy['confidence'] *= Config.ENSEMBLE_WEIGHT_OCR
                ocr_det_copy['method'] = 'OCR_Added'
                merged_results.append(ocr_det_copy)
        
        return merged_results
    
    def group_nearby_detections(self, detections, threshold=50):
        """Group nearby character detections into words"""
        if not detections:
            return []
        
        # Sort by x-coordinate
        sorted_dets = sorted(detections, key=lambda x: x['bbox'][0])
        
        groups = []
        current_group = [sorted_dets[0]]
        
        for i in range(1, len(sorted_dets)):
            current_det = sorted_dets[i]
            last_det = current_group[-1]
            
            # Check horizontal distance
            distance = current_det['bbox'][0] - last_det['bbox'][2]
            
            # Check vertical overlap (same line)
            y_overlap = min(current_det['bbox'][3], last_det['bbox'][3]) - max(current_det['bbox'][1], last_det['bbox'][1])
            total_height = max(current_det['bbox'][3], last_det['bbox'][3]) - min(current_det['bbox'][1], last_det['bbox'][1])
            vertical_overlap_ratio = y_overlap / total_height if total_height > 0 else 0
            
            # Group if close horizontally and on same line
            if distance < threshold and vertical_overlap_ratio > 0.5:
                current_group.append(current_det)
            else:
                groups.append(current_group)
                current_group = [current_det]
        
        groups.append(current_group)
        return groups
    
    def calculate_iou(self, box1, box2):
        """Calculate Intersection over Union (IoU) of two bounding boxes"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        if x2 <= x1 or y2 <= y1:
            return 0.0
        
        intersection = (x2 - x1) * (y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection
        
        return intersection / union if union > 0 else 0.0
    
    def validate_serial_patterns(self, detections):
        """Validate detected text against known serial patterns"""
        validated_serials = []
        
        for detection in detections:
            text = detection['text'].upper()
            clean_text = re.sub(r'[^A-Z0-9]', '', text)
            
            # Check if matches any pattern
            for pattern in self.serial_patterns:
                if re.match(pattern, clean_text):
                    detection_copy = detection.copy()
                    detection_copy['validated'] = True
                    detection_copy['pattern_matched'] = pattern
                    detection_copy['clean_text'] = clean_text
                    validated_serials.append(detection_copy)
                    break
        
        return validated_serials
    
    def predict_image(self, image_path, visualize=True):
        """Complete hybrid prediction pipeline"""
        print(f"🔍 Processing: {Path(image_path).name}")
        
        # Load image
        if isinstance(image_path, str):
            image = cv2.imread(image_path)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        else:
            image_rgb = image_path
        
        # Preprocess for OCR
        preprocessed = self.preprocess_image_for_ocr(image_rgb)
        
        # Run both detection methods
        print("   🔤 Running EasyOCR detection...")
        ocr_detections = self.run_easyocr_detection(image_rgb)
        
        print("   🎯 Running YOLO detection...")
        yolo_detections = self.run_yolo_detection(image_rgb)
        
        # Merge detections
        print("   🔗 Merging detection results...")
        merged_detections = self.merge_detections(ocr_detections, yolo_detections, image_rgb.shape)
        
        # Validate against patterns
        print("   ✅ Validating serial patterns...")
        validated_serials = self.validate_serial_patterns(merged_detections)
        
        # Results summary
        results = {
            'image_path': str(image_path),
            'ocr_detections': len(ocr_detections),
            'yolo_detections': len(yolo_detections),
            'merged_detections': merged_detections,
            'validated_serials': validated_serials,
            'best_prediction': validated_serials[0] if validated_serials else (merged_detections[0] if merged_detections else None)
        }
        
        print(f"   📊 Results: {len(ocr_detections)} OCR + {len(yolo_detections)} YOLO → {len(validated_serials)} validated serials")
        
        if visualize and results['best_prediction']:
            self.visualize_hybrid_results(image_rgb, results)
        
        return results
    
    def visualize_hybrid_results(self, image, results):
        """Visualize hybrid detection results"""
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # Original image
        axes[0].imshow(image)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # OCR + YOLO detections
        image_with_all = image.copy()
        
        # Draw OCR detections in blue
        for det in results['merged_detections']:
            if 'OCR' in det['method']:
                bbox = det['bbox']
                rect = Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1], 
                               linewidth=2, edgecolor='blue', facecolor='none')
                axes[1].add_patch(rect)
        
        # Draw YOLO detections in red
        for det in results['merged_detections']:
            if 'YOLO' in det['method']:
                bbox = det['bbox']
                rect = Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1], 
                               linewidth=2, edgecolor='red', facecolor='none')
                axes[1].add_patch(rect)
        
        axes[1].imshow(image_with_all)
        axes[1].set_title('All Detections (Blue=OCR, Red=YOLO)')
        axes[1].axis('off')
        
        # Final validated result
        image_final = image.copy()
        for det in results['validated_serials']:
            bbox = det['bbox']
            rect = Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1], 
                           linewidth=3, edgecolor='green', facecolor='none')
            axes[2].add_patch(rect)
            
            # Add text label
            axes[2].text(bbox[0], bbox[1]-10, f"{det['clean_text']} ({det['confidence']:.2f})", 
                        fontsize=12, color='green', fontweight='bold')
        
        axes[2].imshow(image_final)
        axes[2].set_title('Validated Serial Numbers')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.savefig(Config.VISUALIZATIONS_DIR / f'hybrid_prediction_{Path(results["image_path"]).stem}.png', 
                   dpi=300, bbox_inches='tight')
        plt.show()
        
        return fig

# Example usage (will work once model is trained)
print("🔬 Initializing Hybrid OCR+YOLO Predictor...")

# You would use the trained model path here
# hybrid_predictor = HybridOCRYOLOPredictor(yolo_model_path="path/to/best.pt")
hybrid_predictor = HybridOCRYOLOPredictor()  # For now, just OCR

print("✅ Hybrid predictor ready!")


In [ ]:
class ModelEvaluator:
    """Comprehensive model evaluation and testing suite"""
    
    def __init__(self, model_path=None, test_data_dir=None):
        self.model_path = model_path
        self.test_data_dir = test_data_dir or Config.DATASET_DIR / 'test'
        self.model = None
        self.evaluation_results = {}
        
        if model_path and Path(model_path).exists():
            self.model = YOLO(model_path)
            print(f"✅ Model loaded for evaluation: {model_path}")
    
    def run_model_validation(self):
        """Run comprehensive model validation"""
        if not self.model:
            print("⚠️ No model loaded for validation")
            return
        
        print("🔍 Running model validation...")
        
        # Run validation on the dataset
        validation_results = self.model.val(
            data=str(Config.DATASET_DIR / 'dataset.yaml'),
            imgsz=Config.IMAGE_SIZE,
            batch=Config.BATCH_SIZE,
            conf=0.3,
            iou=0.7,
            save_json=True,
            plots=True
        )
        
        self.evaluation_results['validation'] = validation_results
        
        # Extract key metrics
        metrics = validation_results.results_dict
        
        print("\\n" + "=" * 60)
        print("📊 MODEL VALIDATION RESULTS")
        print("=" * 60)
        print(f"📈 mAP@0.5: {metrics.get('metrics/mAP50(B)', 'N/A'):.4f}")
        print(f"📈 mAP@0.5:0.95: {metrics.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
        print(f"🎯 Precision: {metrics.get('metrics/precision(B)', 'N/A'):.4f}")
        print(f"🎯 Recall: {metrics.get('metrics/recall(B)', 'N/A'):.4f}")
        print(f"⚡ Inference Speed: {metrics.get('speed/inference', 'N/A'):.2f} ms")
        print("=" * 60)
        
        return validation_results
    
    def test_on_sample_images(self, num_samples=5):
        """Test model on sample images with detailed analysis"""
        if not self.model:
            print("⚠️ No model loaded for testing")
            return
        
        # Get sample test images
        test_images_dir = self.test_data_dir / 'images'
        if not test_images_dir.exists():
            test_images_dir = Config.DATASET_DIR / 'val' / 'images'
        
        sample_images = list(test_images_dir.glob('*.jpg'))[:num_samples]
        
        if not sample_images:
            print("⚠️ No test images found")
            return
        
        print(f"🔍 Testing on {len(sample_images)} sample images...")
        
        test_results = []
        
        for img_path in sample_images:
            print(f"   📸 Testing: {img_path.name}")
            
            # Run inference
            results = self.model(str(img_path), conf=0.3, iou=0.7)
            
            # Load ground truth
            label_path = img_path.parent.parent / 'labels' / f"{img_path.stem}.txt"
            ground_truth = self.load_ground_truth(label_path)
            
            # Analyze results
            analysis = self.analyze_single_prediction(results, ground_truth, img_path)
            test_results.append(analysis)
            
            # Visualize
            self.visualize_prediction(img_path, results, ground_truth)
        
        # Aggregate results
        self.create_test_summary(test_results)
        
        return test_results
    
    def load_ground_truth(self, label_path):
        """Load ground truth annotations"""
        annotations = []
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    values = line.strip().split()
                    if len(values) == 5:
                        class_id = int(values[0])
                        center_x = float(values[1])
                        center_y = float(values[2])
                        width = float(values[3])
                        height = float(values[4])
                        
                        annotations.append({
                            'class_id': class_id,
                            'center_x': center_x,
                            'center_y': center_y,
                            'width': width,
                            'height': height,
                            'character': Config.ID_TO_CHAR.get(class_id, '?')
                        })
        return annotations
    
    def analyze_single_prediction(self, results, ground_truth, img_path):
        """Analyze prediction vs ground truth for single image"""
        analysis = {
            'image_path': str(img_path),
            'ground_truth_count': len(ground_truth),
            'predicted_count': 0,
            'correct_predictions': 0,
            'false_positives': 0,
            'false_negatives': len(ground_truth),
            'precision': 0.0,
            'recall': 0.0,
            'f1_score': 0.0,
            'predicted_characters': [],
            'ground_truth_characters': [gt['character'] for gt in ground_truth]
        }
        
        if not results or not results[0].boxes:
            return analysis
        
        boxes = results[0].boxes
        predictions = []
        
        for box in boxes:
            class_id = int(box.cls[0].cpu().numpy())
            confidence = float(box.conf[0].cpu().numpy())
            character = Config.ID_TO_CHAR.get(class_id, '?')
            
            predictions.append({
                'class_id': class_id,
                'character': character,
                'confidence': confidence
            })
        
        analysis['predicted_count'] = len(predictions)
        analysis['predicted_characters'] = [p['character'] for p in predictions]
        
        # Calculate metrics (simplified - just character-level accuracy)
        gt_chars = set(analysis['ground_truth_characters'])
        pred_chars = set(analysis['predicted_characters'])
        
        true_positives = len(gt_chars.intersection(pred_chars))
        false_positives = len(pred_chars - gt_chars)
        false_negatives = len(gt_chars - pred_chars)
        
        analysis['correct_predictions'] = true_positives
        analysis['false_positives'] = false_positives
        analysis['false_negatives'] = false_negatives
        
        # Calculate precision, recall, F1
        if len(pred_chars) > 0:
            analysis['precision'] = true_positives / len(pred_chars)
        if len(gt_chars) > 0:
            analysis['recall'] = true_positives / len(gt_chars)
        if analysis['precision'] + analysis['recall'] > 0:
            analysis['f1_score'] = 2 * (analysis['precision'] * analysis['recall']) / (analysis['precision'] + analysis['recall'])
        
        return analysis
    
    def visualize_prediction(self, img_path, results, ground_truth):
        """Visualize prediction vs ground truth"""
        # Load image
        image = cv2.imread(str(img_path))
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image_rgb.shape[:2]
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        
        # Ground truth visualization
        gt_image = image_rgb.copy()
        for gt in ground_truth:
            # Convert normalized coordinates to pixel coordinates
            center_x = gt['center_x'] * w
            center_y = gt['center_y'] * h
            width = gt['width'] * w
            height = gt['height'] * h
            
            x1 = int(center_x - width / 2)
            y1 = int(center_y - height / 2)
            x2 = int(center_x + width / 2)
            y2 = int(center_y + height / 2)
            
            rect = Rectangle((x1, y1), x2-x1, y2-y1, 
                           linewidth=2, edgecolor='green', facecolor='none')
            axes[0].add_patch(rect)
            axes[0].text(x1, y1-5, gt['character'], fontsize=12, color='green', fontweight='bold')
        
        axes[0].imshow(gt_image)
        axes[0].set_title('Ground Truth')
        axes[0].axis('off')
        
        # Prediction visualization
        pred_image = image_rgb.copy()
        if results and results[0].boxes:
            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                class_id = int(box.cls[0].cpu().numpy())
                confidence = float(box.conf[0].cpu().numpy())
                character = Config.ID_TO_CHAR.get(class_id, '?')
                
                rect = Rectangle((x1, y1), x2-x1, y2-y1, 
                               linewidth=2, edgecolor='red', facecolor='none')
                axes[1].add_patch(rect)
                axes[1].text(x1, y1-5, f"{character} ({confidence:.2f})", 
                           fontsize=10, color='red', fontweight='bold')
        
        axes[1].imshow(pred_image)
        axes[1].set_title('Predictions')
        axes[1].axis('off')
        
        plt.suptitle(f'Evaluation: {img_path.name}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(Config.VISUALIZATIONS_DIR / f'eval_{img_path.stem}.png', 
                   dpi=300, bbox_inches='tight')
        plt.show()
    
    def create_test_summary(self, test_results):
        """Create comprehensive test summary"""
        if not test_results:
            return
        
        # Aggregate metrics
        total_gt = sum(r['ground_truth_count'] for r in test_results)
        total_pred = sum(r['predicted_count'] for r in test_results)
        total_correct = sum(r['correct_predictions'] for r in test_results)
        total_fp = sum(r['false_positives'] for r in test_results)
        total_fn = sum(r['false_negatives'] for r in test_results)
        
        overall_precision = total_correct / total_pred if total_pred > 0 else 0
        overall_recall = total_correct / total_gt if total_gt > 0 else 0
        overall_f1 = 2 * (overall_precision * overall_recall) / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0
        
        # Create summary visualization
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('📊 Model Evaluation Summary', fontsize=16, fontweight='bold')
        
        # 1. Per-image metrics
        images = [Path(r['image_path']).stem for r in test_results]
        f1_scores = [r['f1_score'] for r in test_results]
        precisions = [r['precision'] for r in test_results]
        recalls = [r['recall'] for r in test_results]
        
        x = range(len(images))
        axes[0, 0].bar(x, f1_scores, alpha=0.7, label='F1 Score')
        axes[0, 0].bar(x, precisions, alpha=0.7, label='Precision')
        axes[0, 0].bar(x, recalls, alpha=0.7, label='Recall')
        axes[0, 0].set_title('Per-Image Metrics')
        axes[0, 0].set_xlabel('Images')
        axes[0, 0].set_ylabel('Score')
        axes[0, 0].legend()
        axes[0, 0].set_xticks(x)
        axes[0, 0].set_xticklabels(images, rotation=45)
        
        # 2. Character distribution accuracy
        all_gt_chars = []
        all_pred_chars = []
        for r in test_results:
            all_gt_chars.extend(r['ground_truth_characters'])
            all_pred_chars.extend(r['predicted_characters'])
        
        char_accuracy = {}
        for char in Config.CHAR_TO_ID.keys():
            gt_count = all_gt_chars.count(char)
            pred_count = all_pred_chars.count(char)
            if gt_count > 0:
                char_accuracy[char] = min(pred_count, gt_count) / gt_count
        
        if char_accuracy:
            chars = list(char_accuracy.keys())
            accuracies = list(char_accuracy.values())
            axes[0, 1].bar(chars, accuracies)
            axes[0, 1].set_title('Character-Level Accuracy')
            axes[0, 1].set_xlabel('Characters')
            axes[0, 1].set_ylabel('Accuracy')
            axes[0, 1].tick_params(axis='x', rotation=45)
        
        # 3. Overall metrics
        metrics_names = ['Precision', 'Recall', 'F1 Score']
        metrics_values = [overall_precision, overall_recall, overall_f1]
        colors = ['lightcoral', 'lightblue', 'lightgreen']
        
        axes[1, 0].bar(metrics_names, metrics_values, color=colors)
        axes[1, 0].set_title('Overall Performance Metrics')
        axes[1, 0].set_ylabel('Score')
        axes[1, 0].set_ylim(0, 1)
        
        # Add value labels on bars
        for i, v in enumerate(metrics_values):
            axes[1, 0].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
        
        # 4. Confusion summary
        confusion_data = [
            ['True Positives', total_correct],
            ['False Positives', total_fp],
            ['False Negatives', total_fn]
        ]
        
        labels = [d[0] for d in confusion_data]
        values = [d[1] for d in confusion_data]
        colors_conf = ['green', 'red', 'orange']
        
        axes[1, 1].pie(values, labels=labels, colors=colors_conf, autopct='%1.1f%%', startangle=90)
        axes[1, 1].set_title('Detection Results Distribution')
        
        plt.tight_layout()
        plt.savefig(Config.VISUALIZATIONS_DIR / 'evaluation_summary.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Print summary
        print("\\n" + "=" * 60)
        print("📊 COMPREHENSIVE EVALUATION SUMMARY")
        print("=" * 60)
        print(f"🔢 Total Ground Truth Characters: {total_gt}")
        print(f"🔢 Total Predicted Characters: {total_pred}")
        print(f"✅ Correct Predictions: {total_correct}")
        print(f"❌ False Positives: {total_fp}")
        print(f"❌ False Negatives: {total_fn}")
        print(f"📈 Overall Precision: {overall_precision:.4f}")
        print(f"📈 Overall Recall: {overall_recall:.4f}")
        print(f"📈 Overall F1 Score: {overall_f1:.4f}")
        print("=" * 60)
        
        # Save detailed report
        report_data = {
            'overall_metrics': {
                'precision': overall_precision,
                'recall': overall_recall,
                'f1_score': overall_f1,
                'total_ground_truth': total_gt,
                'total_predicted': total_pred,
                'correct_predictions': total_correct,
                'false_positives': total_fp,
                'false_negatives': total_fn
            },
            'per_image_results': test_results,
            'character_accuracy': char_accuracy
        }
        
        report_path = Config.RESULTS_DIR / 'evaluation_report.json'
        with open(report_path, 'w') as f:
            json.dump(report_data, f, indent=2)
        
        print(f"📄 Detailed report saved: {report_path}")
        
        return report_data

# Example usage (requires trained model)
print("📊 Model Evaluation Suite Ready")
print("💡 To run evaluation, load a trained model:")
print("   evaluator = ModelEvaluator(model_path='path/to/best.pt')")
print("   evaluator.run_model_validation()")
print("   evaluator.test_on_sample_images(num_samples=10)")


In [ ]:
# =============================================================================
# COMPREHENSIVE PIPELINE SUMMARY
# =============================================================================

print("🚀 YOLOv8 CHARACTER DETECTION PIPELINE COMPLETE!")
print("=" * 80)
print()

# Pipeline Summary
pipeline_summary = {
    'data_analysis': {
        'total_images': len(list(Config.INPUT_IMAGES_DIR.glob('*.jpg'))),
        'character_classes': len(Config.CHAR_TO_ID),
        'serial_patterns': len(Config.SERIAL_PATTERNS),
        'status': '✅ Complete'
    },
    'dataset_conversion': {
        'format': 'YOLO',
        'splits': 'Train/Val/Test',
        'augmentation': 'Rotation + Brightness/Contrast',
        'status': '✅ Complete'
    },
    'model_training': {
        'architecture': 'YOLOv8',
        'epochs': Config.EPOCHS,
        'optimization': 'AdamW + Mixed Precision',
        'status': '⏳ Ready to Train'
    },
    'hybrid_inference': {
        'methods': 'YOLO + EasyOCR',
        'ensemble': 'Weighted Combination',
        'validation': 'Pattern Matching',
        'status': '✅ Complete'
    },
    'evaluation': {
        'metrics': 'mAP, Precision, Recall, F1',
        'visualization': 'Comprehensive Dashboards',
        'reporting': 'JSON + Plots',
        'status': '✅ Complete'
    }
}

print("📊 PIPELINE COMPONENTS:")
for component, details in pipeline_summary.items():
    print(f"\\n🔧 {component.upper().replace('_', ' ')}:")
    for key, value in details.items():
        if key == 'status':
            print(f"   {key.capitalize()}: {value}")
        else:
            print(f"   {key.capitalize()}: {value}")

print("\\n" + "=" * 80)
print("🎯 ACHIEVEMENT HIGHLIGHTS:")
print("=" * 80)

achievements = [
    "✅ Modular, production-ready codebase with proper error handling",
    "✅ Comprehensive data analysis with character distribution insights",
    "✅ Smart dataset splitting maintaining pattern distribution",
    "✅ Advanced data augmentation with precise coordinate transformation",
    "✅ Optimized YOLOv8 training pipeline with monitoring",
    "✅ Hybrid OCR+YOLO approach for robust character detection",
    "✅ Real-time inference with pattern validation",
    "✅ Comprehensive evaluation suite with detailed metrics",
    "✅ Professional visualizations and reporting",
    "✅ Multiple export formats for production deployment"
]

for achievement in achievements:
    print(achievement)

print("\\n" + "=" * 80)
print("🚀 NEXT STEPS & RECOMMENDATIONS:")
print("=" * 80)

next_steps = [
    "1. 🎯 TRAINING: Run the training pipeline on your 236 annotated images",
    "2. 🔧 OPTIMIZATION: Fine-tune hyperparameters based on validation results",
    "3. 📊 EVALUATION: Test on additional unseen images for real-world performance",
    "4. 🚀 DEPLOYMENT: Export to ONNX/TensorRT for production inference",
    "5. 📈 SCALING: Add more diverse images to improve generalization",
    "6. 🔄 ITERATION: Implement active learning for continuous improvement",
    "7. 🌐 INTEGRATION: Build REST API or web interface for easy usage",
    "8. 📱 MOBILE: Consider mobile optimization for edge deployment",
    "9. 🔒 VALIDATION: Implement robust QA pipeline for production use",
    "10. 📚 DOCUMENTATION: Create user manuals and API documentation"
]

for step in next_steps:
    print(step)

print("\\n" + "=" * 80)
print("💡 TECHNICAL INNOVATIONS IMPLEMENTED:")
print("=" * 80)

innovations = [
    "🧠 Smart pattern-aware dataset splitting for balanced training",
    "🔄 Precise coordinate transformation during rotation augmentation",
    "🤝 Intelligent fusion of YOLO detection with OCR validation",
    "📊 Real-time character frequency analysis for class balancing",
    "🎯 Multi-pattern serial number validation with OCR correction",
    "📈 Comprehensive evaluation with character-level accuracy metrics",
    "🔧 Modular architecture enabling easy extension and customization",
    "⚡ GPU-optimized training with automatic mixed precision",
    "📱 Multiple export formats for various deployment scenarios",
    "🎨 Rich visualization suite for model interpretation and debugging"
]

for innovation in innovations:
    print(innovation)

print("\\n" + "=" * 80)
print("📁 OUTPUT STRUCTURE:")
print("=" * 80)

output_structure = f\"\"\"
📁 {Config.OUTPUT_DIR}/
├── 📊 dataset/                 # YOLO format dataset
│   ├── train/images & labels
│   ├── val/images & labels  
│   ├── test/images & labels
│   └── dataset.yaml
├── 🔄 augmented_data/          # Augmented dataset
│   ├── train/images & labels
│   ├── val/images & labels
│   └── dataset.yaml
├── 🤖 models/                  # Trained models
│   └── character_detection_aug/
│       ├── weights/best.pt
│       ├── weights/last.pt
│       └── results.csv
├── 📊 results/                 # Evaluation results
│   └── evaluation_report.json
└── 🎨 visualizations/          # All plots and visualizations
    ├── dataset_analysis.png
    ├── training_dashboard.png
    ├── augmentation_samples.png
    └── evaluation_summary.png
\"\"\"

print(output_structure)

print("\\n" + "=" * 80)
print("🎉 CONGRATULATIONS!")
print("You now have a complete, production-ready YOLOv8 character detection")
print("pipeline with hybrid OCR integration and comprehensive evaluation!")
print("=" * 80)

# Save pipeline configuration for future reference
pipeline_config = {
    'creation_date': datetime.now().isoformat(),
    'pipeline_version': '2.0_hybrid_yolo_ocr',
    'configuration': {
        'input_images': len(list(Config.INPUT_IMAGES_DIR.glob('*.jpg'))),
        'character_classes': Config.CHAR_TO_ID,
        'serial_patterns': Config.SERIAL_PATTERNS,
        'training_params': {
            'model_size': Config.YOLO_MODEL_SIZE,
            'epochs': Config.EPOCHS,
            'batch_size': Config.BATCH_SIZE,
            'image_size': Config.IMAGE_SIZE
        },
        'augmentation_params': {
            'rotation_angles': Config.ROTATION_ANGLES,
            'brightness_range': Config.BRIGHTNESS_RANGE,
            'contrast_range': Config.CONTRAST_RANGE
        }
    },
    'pipeline_summary': pipeline_summary
}

config_path = Config.OUTPUT_DIR / 'pipeline_configuration.json'
with open(config_path, 'w') as f:
    json.dump(pipeline_config, f, indent=2)

print(f"📄 Pipeline configuration saved: {config_path}")
